In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import numpy as np
import json
import glob
import os
import joblib
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import tqdm.notebook as tqdm

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

# from ST_GCN_model import LateFusion_STGCN
from ST_GCN_v2 import LateFusion_STGCN


In [ ]:
# Configuration
F_DATA_PATH        = 'split_data/front_view'  # thư mục chứa các file .npz        
L_DATA_PATH        = 'split_data/left_view'   # thư mục chứa các file .npz
R_DATA_PATH        = 'split_data/right_view'  # thư mục chứa các file .npz
LABEL_MAP_PATH   = 'Logs/label_map.json'  # đường dẫn đến file label_map.json
BATCH_SIZE       = 128
EPOCHS           = 100
LEARNING_RATE    = 1e-3
DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cpu


In [ ]:
# 1. Load Files and Labels
with open(LABEL_MAP_PATH, 'r', encoding='utf-8') as f:
    label_map = json.load(f)

train_files = glob.glob(os.path.join(F_DATA_PATH, 'train', '**', '*.npz'), recursive=True)
val_files   = glob.glob(os.path.join(F_DATA_PATH, 'val', '**', '*.npz'), recursive=True)


class VSL_GCN_Dataset(Dataset):
    def __init__(self, file_list):
        self.files = file_list
        # Define the parent map for bones (MediaPipe Hand)
        self.parents = [0, 0, 1, 2, 3, 0, 5, 6, 7, 0, 9, 10, 11, 0, 13, 14, 15, 0, 17, 18, 19]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        f_path = self.files[idx]
        
        # 1. Load raw sequence (assuming 60 frames, 64 features)
        # We need to extract the 42 keypoints (21 left, 21 right)
        # Note: Adjust the indexing [:42*3] based on how your .npz is saved!
        raw_data = np.load(f_path)["sequence"].astype(np.float32) # (60, 64)
        
        # Reshape to (Time, Nodes, Channels) -> (60, 42, 3)
        # This assumes your first 126 features are (x,y,z * 42)
        # If your data is 2D, adjust channels to 2.
        # Here we slice and reshape.
        joints = raw_data[:, :126].reshape(60, 42, 3) 
        
        # Transpose to ST-GCN format: (Channels, Time, Nodes) -> (3, 60, 42)
        x_joint = torch.from_numpy(joints).permute(2, 0, 1)

        if self.is_train:
            # 0.005 is roughly 0.5% coordinate shift - enough to 
            # prevent memorization without ruining the sign shape.
            std = 0.005 
            noise = torch.randn_like(x_joint) * std
            x_joint = x_joint + noise

        # 2. Calculate Bone Stream (Relative distance)
        x_bone = torch.zeros_like(x_joint)
        for i in range(21):
            # Left Hand
            x_bone[:, :, i] = x_joint[:, :, i] - x_joint[:, :, self.parents[i]]
            # Right Hand
            x_bone[:, :, i+21] = x_joint[:, :, i+21] - x_joint[:, :, self.parents[i]+21]

        # 3. Calculate Motion Stream (Temporal difference)
        x_motion = torch.zeros_like(x_joint)
        x_motion[:, 1:, :] = x_joint[:, 1:, :] - x_joint[:, :-1, :]

        label = int(np.load(f_path)["label"])
        
        return x_joint, x_bone, x_motion, label

# 3. Simplified DataLoaders
# Batch size can be larger for GCN (e.g., 64 or 128)

train_set = VSL_GCN_Dataset(train_files, is_train=True)
val_set = VSL_GCN_Dataset(val_files, is_train=False)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=12, pin_memory=True)
val_loader   = DataLoader(val_set, batch_size=128, shuffle=False, num_workers=12, pin_memory=True)         

Fitting Scaler on training samples...
Scaler saved to vsl_scaler.pkl


In [ ]:

class EarlyStopping:
    def __init__(self, patience=7, min_delta=0, path='best_vsl_model.pth'):
        self.patience = patience
        self.min_delta = min_delta
        self.path = path
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.save_checkpoint(model)
            self.counter = 0

    def save_checkpoint(self, model):
        torch.save(model.state_dict(), self.path)

In [5]:
def plot_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Plot 1: Accuracy
    ax1.plot(history['accuracy'], label='Train Accuracy', color='tab:blue')
    ax1.plot(history['val_accuracy'], label='Val Accuracy', color='tab:orange')
    ax1.set_title('Training and Validation Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.set_ylim([0, 1.05]) 
    ax1.grid(True)
    ax1.legend(loc='lower right')

    # Plot 2: Loss 
    ax2.plot(history['loss'], label='Train Loss', color='tab:blue')
    ax2.plot(history['val_loss'], label='Val Loss', color='tab:orange')
    ax2.set_title('Training and Validation Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.grid(True)
    ax2.legend(loc='upper right')

    plt.tight_layout()
    plt.show()

In [ ]:
history = {
    'accuracy': [],
    'val_accuracy': [],
    'loss': [],
    'val_loss': [],
    'val_top1': [], 'val_top5': [] # Track these per epoch

}


model = LateFusion_STGCN(num_classes=len(label_map)).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=2e-2)
criterion = nn.CrossEntropyLoss(label_smoothing=0.15)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6)
steps_per_epoch = len(train_loader)
# scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=1e-3, steps_per_epoch=steps_per_epoch, epochs=EPOCHS,pct_start=0.1, div_factor=25, final_div_factor=1000)
grad_scaler = GradScaler() # For FP16 Mixed Precision
early_stopper = EarlyStopping(patience=10, path='best_stgcn_model.pth')


print("Starting Training...")

for epoch in range(1, EPOCHS + 1):
    # --- TRAINING ---
    model.train()
    train_loss, train_correct = 0.0, 0
    
    pbar = tqdm.tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    for x_joint, x_bone, x_motion, labels in pbar:
        x_joint, x_bone, x_motion, labels = x_joint.to(DEVICE, non_blocking=True), x_bone.to(DEVICE, non_blocking=True), x_motion.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()

        with autocast(device_type='cuda' if DEVICE.type == 'cuda' else 'cpu'):
            outputs = model(x_joint, x_bone, x_motion)
            loss = criterion(outputs, labels)

        grad_scaler.scale(loss).backward()
        
        # Unscale for gradient clipping
        grad_scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        grad_scaler.step(optimizer)
        grad_scaler.update()

        # scheduler.step()

        train_loss += loss.item() * x_joint.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()

    # --- VALIDATION ---
    model.eval()
    val_loss, val_correct = 0.0, 0
    val_top1_correct = 0
    val_top5_correct = 0
    with torch.no_grad():
        for x_joint, x_bone, x_motion, labels in val_loader:
            x_joint, x_bone, x_motion, labels = x_joint.to(DEVICE, non_blocking=True), x_bone.to(DEVICE, non_blocking=True), x_motion.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            with autocast(device_type='cuda' if DEVICE.type == 'cuda' else 'cpu'):
                outputs = model(x_joint, x_bone, x_motion)
                loss = criterion(outputs, labels)
            
            val_loss += loss.item() * x_joint.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()

                        # Top-K Calculation
            _, top5_indices = outputs.topk(5, dim=1)
            expanded_labels = labels.view(-1, 1).expand_as(top5_indices)
            correct_mask = (top5_indices == expanded_labels)
            
            val_top1_correct += correct_mask[:, 0].sum().item()
            val_top5_correct += correct_mask.sum().item()

    # --- METRICS CALCULATION ---
    t_loss = train_loss / len(train_files)
    t_acc  = (train_correct / len(train_files)) 
    v_loss = val_loss / len(val_files)
    v_acc  = (val_correct / len(val_files))
    v_top1_acc = (val_top1_correct / len(val_files))
    v_top5_acc = (val_top5_correct / len(val_files))

    # Save to history
    history['loss'].append(t_loss)
    history['accuracy'].append(t_acc)
    history['val_loss'].append(v_loss)
    history['val_accuracy'].append(v_acc)

    print(f"Epoch {epoch:02d}: "
          f"Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.4f} | "
          f"Val Loss: {v_loss:.4f} | Val Acc: {v_acc:.4f}")
    print(f"Top-1 Accuracy: {v_top1_acc:.4f} | Top-5 Accuracy: {v_top5_acc:.4f} ")
    
# Inside your epoch loop, after validation
    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(v_loss)
    new_lr = optimizer.param_groups[0]['lr']

    if new_lr < old_lr:
        print(f"📉 Learning rate reduced from {old_lr:.6f} to {new_lr:.6f}")
    
    early_stopper(v_loss, model)
    if early_stopper.early_stop:
        print("Early stopping triggered. Training halted.")
        break
    
    # Save best model
plot_history(history)

Starting Training...


C:\Users\ASUS\AppData\Local\Temp\ipykernel_25172\1271988991.py:13: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  grad_scaler = GradScaler() # For FP16 Mixed Precision


Epoch 1/50 [Train]:   0%|          | 0/2500 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
test_files = glob.glob(os.path.join(F_DATA_PATH, 'test', '**', '*.npz'), recursive=True)
test_loader = DataLoader(VSL_GCN_Dataset(test_files), batch_size=BATCH_SIZE, shuffle=False)

model = LateFusion_STGCN(num_classes=len(label_map)).to(DEVICE)
model.load_state_dict(torch.load('best_vsl_model.pth'))
model.eval()

test_correct = 0
print("Evaluating on Test Set...")
with torch.no_grad():
    for x_f, x_l, x_r, labels in tqdm.tqdm(test_loader, desc="Testing"):
        x_f, x_l, x_r, labels = x_f.to(DEVICE), x_l.to(DEVICE), x_r.to(DEVICE), labels.to(DEVICE)
        with autocast(device_type='cuda' if DEVICE.type == 'cuda' else 'cpu'):
            outputs = model(x_f, x_l, x_r)

        test_correct += (outputs.argmax(1) == labels).sum().item()

final_acc = (test_correct / len(test_files)) * 100
print(f"Final Test Accuracy: {final_acc:.2f}%")